<div style="text-align: center; padding: 40px 0; border-bottom: 2px solid #e0e0e0; margin-bottom: 20px;">
  <h1 style="font-size: 2.5em; font-weight: 700; color: #000000; margin: 0;">
    🇫🇷 France Public Transport
  </h1>
  <h2 style="font-size: 1.4em; font-weight: 300; color: #555; margin: 10px 0 0 0;">
    GTFS and NeTEx Dataset Analysis — 2026
  </h2>
  <p style="color: #999; margin-top: 12px; font-size: 0.9em;">
    Source: transport.data.gouv.fr · Provider: SNCF Voyageurs · Format: GTFS & NeTEx
  </p>
</div>

## Data Source

**France GTFS Dataset**
- Provider: French National Access Point (transport.data.gouv.fr)
- Publisher: SNCF Voyageurs
- Format: GTFS
- Dataset page: https://transport.data.gouv.fr/datasets/horaires-sncf
- Download URL: https://eu.ftp.opendatasoft.com/sncf/plandata/Export_OpenData_SNCF_GTFS_NewTripId.zip
- Accessed: April 2026
- License: ODbL (Open Database Licence)

**France NeTEx Dataset**
- Provider: French National Access Point (transport.data.gouv.fr)
- Publisher: SNCF Voyageurs
- Format: NeTEx
- Dataset page: https://transport.data.gouv.fr/datasets/horaires-sncf
- Download URL: https://eu.ftp.opendatasoft.com/sncf/plandata/export-opendata-sncf-netex.zip
- Accessed: April 2026
- License: ODbL (Open Database Licence)

*No registration needed for downloading the data*

## Table of Contents

### Part 1: France GTFS Exploration
- Comment on the GTFS ZIP contents
- Helper function to read GTFS tables from the ZIP
- Load the main GTFS tables for the first exploration
- Inspect the loaded GTFS tables
- Check the stop hierarchy
- Define the GTFS station-level subset
- Check the quality of the GTFS station-level subset
- Explore the GTFS line-level fields
- Check repeated public line labels in GTFS
- Explore the GTFS calendar_dates table
- Check active dates per service_id
- Check the distribution of active dates per service_id
- Link the GTFS calendar to trips
- Inspect the unmatched service_id
- Check the GTFS route types
- Summary: France GTFS

### Part 2: France NeTEx Exploration
- Inspect the basic structure of the France NeTEx XML
- Inspect the first level of the France NeTEx structure
- Peeking at the raw XML without parsing
- 2.1 Extracting Stations from the NeTEx XML File
  - Step 1: Extract the XML file to disk
  - Step 2: Pull out only the StopPlace lines
  - Extracting full StopPlace blocks with sed
  - Parsing the StopPlace blocks into a DataFrame
  - Inspecting the NeTEx StopPlace table
- 2.2 Extracting Lines from the NeTEx XML File
  - Parsing the Line blocks into a DataFrame
  - Checking Public Code Uniqueness
- 2.3 Extracting Calendar from the NeTEx XML File
  - Extracting UicOperatingPeriod blocks
  - Parsing the UicOperatingPeriod blocks into a DataFrame
  - Extracting DayType blocks

### Part 3: GTFS and NeTEx Comparison
- 3.1 Station-Level Comparison
  - Preparing the coordinates for matching
  - Conclusion: Station Comparison
- 3.2 Line-Level Comparison
  - Verifying ID overlap between GTFS and NeTEx
  - Joining GTFS Routes and NeTEx Lines
  - Conclusion: Line Comparison
- 3.3 Calendar-Level Comparison
  - Preparing the GTFS Calendar Side
  - Comparing GTFS and NeTEx Calendar Patterns
  - Conclusion: Calendar Comparison
- 3.4 Summary: France GTFS vs NeTEx Comparison

# Part 1: France GTFS Exploration

All third-party and standard-library imports used in this notebook are consolidated here.

In [1]:
from pathlib import Path
import zipfile
import pandas as pd
import xml.etree.ElementTree as ET
import os
import subprocess
from haversine import haversine, Unit

In [2]:
# Set path

data_dir = Path("data")

gtfs_zip_path = data_dir / "Export_OpenData_SNCF_GTFS_NewTripId.zip"

print("GTFS exists:", gtfs_zip_path.exists(), gtfs_zip_path)

GTFS exists: True data/Export_OpenData_SNCF_GTFS_NewTripId.zip


In [3]:
# Inspect GTFS zip file

with zipfile.ZipFile(gtfs_zip_path, "r") as z:
    gtfs_files = pd.DataFrame([
        {
            "name": info.filename,
            "size_mb": round(info.file_size / (1024 * 1024), 2)
        }
        for info in z.infolist()
    ])

display(gtfs_files.sort_values("size_mb", ascending=False))

,name,size_mb
5,stop_times.txt,70.19
6,trips.txt,8.72
0,calendar_dates.txt,7.57
2,stops.txt,0.69
4,routes.txt,0.07
1,agency.txt,0.00
3,transfers.txt,0.00
7,feed_info.txt,0.00


## Observation
### Comment on the GTFS ZIP contents

The France GTFS ZIP contains the main files needed for the first exploration, including `stops`, `routes`, `trips`, `stop_times`, and `calendar_dates`.

<div style="background-color: #d0e8ff; border-left: 5px solid #1a73e8; padding: 12px 16px; border-radius: 4px; margin: 10px 0;">
  <strong>ℹ️ Important</strong><br>
  One important difference compared with some earlier country cases is that there is no <code>calendar.txt</code> file here. This means the service validity in this feed is likely defined only through <code>calendar_dates</code>, which means GTFS does NOT use weekly rules, it is exception-based only.
</div>


## Helper function to read GTFS tables from the ZIP

I use the same helper-function approach as in the earlier country notebooks.

In [4]:
def read_gtfs_from_zip(zip_path: Path, filename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()

        # If the exact file name is present, use it directly
        if filename in names:
            target = filename
        else:
            # Otherwise, search for it anywhere inside the ZIP
            matches = [n for n in names if n.lower().endswith("/" + filename.lower())]

            if not matches:
                raise FileNotFoundError(f"{filename} not found in ZIP. Example entries: {names[:20]}")
            if len(matches) > 1:
                raise FileNotFoundError(f"Multiple matches found for {filename}: {matches}")

            target = matches[0]

        with z.open(target) as f:
            return pd.read_csv(f, low_memory=False)

## Load the main GTFS tables for the first exploration

I now load the smaller GTFS tables that are needed for the first inspection:
- `stops`
- `routes`
- `trips`
- `calendar_dates`

I leave `stop_times` for later because it is much larger and not needed yet for the first structural checks.

In [5]:
stops = read_gtfs_from_zip(gtfs_zip_path, "stops.txt")
routes = read_gtfs_from_zip(gtfs_zip_path, "routes.txt")
trips = read_gtfs_from_zip(gtfs_zip_path, "trips.txt")
calendar_dates = read_gtfs_from_zip(gtfs_zip_path, "calendar_dates.txt")

print("stops:", stops.shape)
print("routes:", routes.shape)
print("trips:", trips.shape)
print("calendar_dates:", calendar_dates.shape)

stops: (8795, 9)
routes: (728, 9)
trips: (52998, 7)
calendar_dates: (441056, 3)


### Comment

The France GTFS feed is quite large.

The `stops` table has 8,795 rows and the `routes` table 728 rows, which is still manageable for the first exploration. But the feed becomes much larger at service level: `trips` already has 52,998 rows, and `calendar_dates` is especially large with 441,056 rows.

This already suggests that the France GTFS feed is much heavier on the operational and calendar side than Luxembourg. It also confirms that the calendar part will need special attention later, especially because the feed uses `calendar_dates` only.

## Inspect the loaded GTFS tables

After checking the table sizes, I now inspect the columns and first rows of the main GTFS tables.

In [6]:
print("stops columns:", list(stops.columns))
display(stops.head())

print("routes columns:", list(routes.columns))
display(routes.head())

print("trips columns:", list(trips.columns))
display(trips.head())

print("calendar_dates columns:", list(calendar_dates.columns))
display(calendar_dates.head())

stops columns: ['stop_id', 'stop_name', 'stop_desc', 'stop_lat', 'stop_lon', 'zone_id', 'stop_url', 'location_type', 'parent_station']


,stop_id,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station
0,StopArea:OCE71043075,FIGUERES-VILAFANT,NaN,42.264581,2.943028,NaN,NaN,1,NaN
1,StopPoint:OCETGV INOUI-71043075,FIGUERES-VILAFANT,NaN,42.264581,2.943028,NaN,NaN,0,StopArea:OCE71043075
2,StopArea:OCE71718010,Barcelone-Sants,NaN,41.378961,2.139834,NaN,NaN,1,NaN
3,StopPoint:OCETGV INOUI-71718010,Barcelone-Sants,NaN,41.378961,2.139834,NaN,NaN,0,StopArea:OCE71718010
4,StopArea:OCE71793000,GIRONA,NaN,41.979371,2.816957,NaN,NaN,1,NaN


routes columns: ['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_desc', 'route_type', 'route_url', 'route_color', 'route_text_color']


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_url,route_color,route_text_color
0,FR:Line::00F2577A-6A87-42E0-95F3-07351E4BC2F6:,1187,P53,Bening - Sarreguemines,NaN,3,NaN,006600,FFFFFF
1,FR:Line::00F7208C-CEBC-4521-A792-6EC3ABB65811:,1187,C30,Saint-Étienne - Roanne,NaN,2,NaN,0749FF,FFFFFF
2,FR:Line::00e52513-07a0-4ce1-8b13-97cab39bdef8:,1187,S3,Ligne S3 BreizhGo,NaN,2,NaN,A45D0C,NaN
3,FR:Line::0128E1D5-9183-4D58-B1CF-F5AA5A64A037:,1187,C6,Marseille - Toulon - Hyeres,NaN,2,NaN,0749FF,FFFFFF
4,FR:Line::0202671B-7107-429E-A37B-473C55E0254C:,1187,C8,Montpellier Saint-Roch - Avignon Centre,NaN,2,NaN,0749FF,FFFFFF


trips columns: ['route_id', 'service_id', 'trip_id', 'trip_headsign', 'direction_id', 'block_id', 'shape_id']


,route_id,service_id,trip_id,trip_headsign,direction_id,block_id,shape_id
0,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,1,OCESN105241F1187_F:NAV:FR:Line::85d3579c-996b-...,105241,0.0,1,NaN
1,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,2,OCESN105342F1187_F:NAV:FR:Line::85d3579c-996b-...,105342,1.0,2,NaN
2,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,3,OCESN105342F1187_F:NAV:FR:Line::85d3579c-996b-...,105342,1.0,3,NaN
3,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,4,OCESN105342F1187_F:NAV:FR:Line::85d3579c-996b-...,105342,1.0,4,NaN
4,FR:Line::85d3579c-996b-4671-89af-839855ede78a:,5,OCESN105347F1187_F:NAV:FR:Line::85d3579c-996b-...,105347,1.0,5,NaN


calendar_dates columns: ['service_id', 'date', 'exception_type']


,service_id,date,exception_type
0,1,20260706,1
1,1,20260707,1
2,1,20260708,1
3,1,20260709,1
4,1,20260710,1


## Comment

The first preview shows that the France GTFS feed follows the usual GTFS structure, but with some clear country-specific details.

In `stops`, the stop hierarchy is already visible: there are station-like records with `location_type = 1` and child stop records with `location_type = 0` linked through `parent_station`.

In `routes`, the main visible public line label seems to be `route_short_name`, while `route_id` is a much longer technical identifier.

In `trips`, the key link fields are present: `route_id`, `service_id`, and `trip_id`.

In `calendar_dates`, the structure is very simple, with only `service_id`, `date`, and `exception_type`. This confirms again that the France feed handles service validity through `calendar_dates` only.

## Check the stop hierarchy

I now inspect the GTFS stop hierarchy by looking at `location_type` and `parent_station`.
.

In [7]:
print("location_type value counts:")
print(stops["location_type"].value_counts(dropna=False))

print("\nparent_station non-null count:")
print(stops["parent_station"].notna().sum())

location_type value counts:
location_type
0    5289
1    3506
Name: count, dtype: int64

parent_station non-null count:
5289


## Comment


The France GTFS stops table is clearly hierarchical.

There are only two `location_type` values here:
- `0` with 5,289 rows
- `1` with 3,506 rows

The `parent_station` non-null count is also 5,289, which matches exactly the number of `location_type = 0` rows. This suggests that the child stop records are linked to parent station records in a very clean way.

So unlike Luxembourg, the France GTFS stop structure is not flat. It is closer to Austria and Switzerland, where station-level and child-stop relationships also mattered.

## Define the GTFS station-level subset

Since the France GTFS stops table is hierarchical, I now define the station-level subset by keeping only stops with `location_type = 1`.

These are the records that are closest to NeTEx `StopPlace`, so they are the right GTFS unit for the later station-level comparison.

In [8]:
gtfs_station_stops = stops[stops["location_type"] == 1].copy()

print("GTFS station-level subset shape:", gtfs_station_stops.shape)
display(gtfs_station_stops.head())

GTFS station-level subset shape: (3506, 9)


,stop_id,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station
0,StopArea:OCE71043075,FIGUERES-VILAFANT,NaN,42.264581,2.943028,NaN,NaN,1,NaN
2,StopArea:OCE71718010,Barcelone-Sants,NaN,41.378961,2.139834,NaN,NaN,1,NaN
4,StopArea:OCE71793000,GIRONA,NaN,41.979371,2.816957,NaN,NaN,1,NaN
6,StopArea:OCE71793150,Portbou,NaN,42.424701,3.158034,NaN,NaN,1,NaN
8,StopArea:OCE80021402,Augsburg Hbf,NaN,48.365400,10.886000,NaN,NaN,1,NaN


## Comment

This subset keeps only the station-level GTFS records.

It reduces the full stops table to the objects that are most suitable for the later station-level comparison with NeTEx.

## Check the quality of the GTFS station-level subset

Now that the station-level GTFS subset is defined, I check whether the core fields are complete.

The main fields for the later station-level comparison are:
- `stop_id`
- `stop_name`
- `stop_lat`
- `stop_lon`

In [9]:
print("Missingness in GTFS station-level core fields:")
print(
    gtfs_station_stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

Missingness in GTFS station-level core fields:
stop_id      0.0
stop_name    0.0
stop_lat     0.0
stop_lon     0.0
dtype: float64


## Comment

The GTFS station-level subset looks very clean. The four core fields needed for the later station-level comparison have no missing values in this subset.

So on the GTFS side, the station-level data is complete and suitable for later comparison with NeTEx.

## Explore the GTFS line-level fields

After checking the stop structure, I now look at the GTFS route table.

The main goal here is to see whether `route_short_name` can be used as the visible public line label later in the GTFS–NeTEx comparison.

In [10]:
print("Missingness in key GTFS route fields:")
print(
    routes[["route_id", "route_short_name", "route_long_name", "route_type"]]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print("\nNumber of routes:", len(routes))
print("Unique route_id:", routes["route_id"].nunique())
print("Unique route_short_name:", routes["route_short_name"].nunique())

display(
    routes[["route_id", "route_short_name", "route_long_name", "route_type"]]
    .head(10)
)

Missingness in key GTFS route fields:
route_id            0.0
route_short_name    0.0
route_long_name     0.0
route_type          0.0
dtype: float64

Number of routes: 728
Unique route_id: 728
Unique route_short_name: 427


,route_id,route_short_name,route_long_name,route_type
0,FR:Line::00F2577A-6A87-42E0-95F3-07351E4BC2F6:,P53,Bening - Sarreguemines,3
1,FR:Line::00F7208C-CEBC-4521-A792-6EC3ABB65811:,C30,Saint-Étienne - Roanne,2
2,FR:Line::00e52513-07a0-4ce1-8b13-97cab39bdef8:,S3,Ligne S3 BreizhGo,2
3,FR:Line::0128E1D5-9183-4D58-B1CF-F5AA5A64A037:,C6,Marseille - Toulon - Hyeres,2
4,FR:Line::0202671B-7107-429E-A37B-473C55E0254C:,C8,Montpellier Saint-Roch - Avignon Centre,2
5,FR:Line::022B77D9-D121-4DCB-B808-FB2F7931866B:,C20,Nantes Cholet Angers Saumur,2
6,FR:Line::02534A5F-903C-454E-A24F-E6E2E23B3CBF:,P13,Ambérieu - Mâcon,2
7,FR:Line::0312E125-B6BC-404C-A7D1-C1600D4CACAF:,K1,Toulouse Matabiau - Brive La Gaillarde,2
8,FR:Line::03398D75-5B60-4588-BC05-9985E599BA20:,R16,Ussel - Bort Les Orgues,3
9,FR:Line::0359c353-2f81-4c49-a269-bc287ff237e0:,L22,22. Limoges - Uzerche - Brive,2


### Comment

The GTFS route table looks very clean. The main route fields have no missing values.
There are 728 route records in total, and all `route_id` values are unique. But `route_short_name` has only 427 unique values, so the same visible line label appears more than once in the feed.

This suggests that, as in earlier country cases, the public line label will likely be more useful for the later comparison than the raw technical `route_id`.

## Check repeated public line labels in GTFS

I now check how often the same `route_short_name` appears in the France GTFS feed.


In [11]:
route_short_name_counts = (
    routes["route_short_name"]
    .value_counts(dropna=False)
    .rename_axis("route_short_name")
    .reset_index(name="n_routes")
)

display(route_short_name_counts.head(20))

print("Number of route_short_name values that appear more than once:",
      (route_short_name_counts["n_routes"] > 1).sum())

,route_short_name,n_routes
0,INCONNU,64
1,P14,7
2,P4,7
3,P3,6
4,P11,6
5,P10,6
6,C1,6
7,P2,6
8,C2,6
9,C10,6


Number of route_short_name values that appear more than once: 107


## Comment

The output shows clearly that many visible GTFS line labels are repeated across several route records.

There are 107 `route_short_name` values that appear more than once, so the same public label is often used by multiple technical `route_id` records. This supports the same idea seen in earlier country cases: for the later GTFS–NeTEx line comparison, the visible public label will likely be more useful than the raw route id.

One special case is `INCONNU`, which appears 64 times. This looks more like a generic placeholder than a meaningful public line label, so it may need special attention later in the line-level comparison.

## Explore the GTFS calendar_dates table

Since the France GTFS feed does not contain `calendar.txt`, I now inspect `calendar_dates` more closely.

The goal is to understand the date range, the number of service ids, and how the exception types are used in this feed.

In [12]:
print("calendar_dates date min:", calendar_dates["date"].min())
print("calendar_dates date max:", calendar_dates["date"].max())
print("unique service_id in calendar_dates:", calendar_dates["service_id"].nunique())

print("\nexception_type value counts:")
print(calendar_dates["exception_type"].value_counts(dropna=False))

calendar_dates date min: 20260426
calendar_dates date max: 20261031
unique service_id in calendar_dates: 12815

exception_type value counts:
exception_type
1    441056
Name: count, dtype: int64


### Comment

The France GTFS calendar structure is very different from the earlier country cases.

The feed covers the period from 2026-04-26 to 2026-10-31, and `calendar_dates` contains 12,815 unique `service_id` values. Also, all 441,056 rows have `exception_type = 1`.

This suggests that `calendar_dates` is not being used here as a small exception table on top of `calendar.txt`. Instead, it seems to contain the full list of active service dates. So for France, the calendar side will likely need to be reconstructed directly from `calendar_dates`.

## Check active dates per service_id

Since the France GTFS feed uses only `calendar_dates`, I now look at how many active dates are linked to each `service_id`.

This helps me understand whether services run for long periods or only on a few dates, and it gives a first idea of the service-pattern structure in the feed.

In [13]:
service_date_counts = (
    calendar_dates.groupby("service_id")
    .size()
    .sort_values(ascending=False)
    .rename("n_active_dates")
    .reset_index()
)

display(service_date_counts.head(20))

print("Min active dates per service_id:", service_date_counts["n_active_dates"].min())
print("Max active dates per service_id:", service_date_counts["n_active_dates"].max())
print("Mean active dates per service_id:", round(service_date_counts["n_active_dates"].mean(), 2))

,service_id,n_active_dates
0,0,189
1,28,181
2,10843,180
3,5161,180
4,4933,180
5,2136,180
6,9858,180
7,12180,180
8,12183,180
9,9652,180


Min active dates per service_id: 1
Max active dates per service_id: 189
Mean active dates per service_id: 34.42


## Comment

The number of active dates per `service_id` varies a lot in the France GTFS feed.

Some service ids are active on almost the full feed period, with up to 189 dates, while others are active on only a few dates. The average is much lower at 34.42 dates per service id, which suggests that many service ids are quite short-lived.

So the France calendar structure looks much more fragmented than a simple regular weekly pattern. This supports the earlier impression that `calendar_dates` is carrying the real service logic in this feed.

## Check the distribution of active dates per service_id

I now look at the distribution of active dates per `service_id`.

This helps me see whether most service ids are short-lived or whether many of them run over long parts of the feed period.

In [14]:
active_date_distribution = (
    service_date_counts["n_active_dates"]
    .value_counts()
    .sort_index()
    .rename_axis("n_active_dates")
    .reset_index(name="n_service_ids")
)

display(active_date_distribution.head(20))
display(active_date_distribution.tail(20))

,n_active_dates,n_service_ids
0,1,169
1,2,626
2,3,637
3,4,567
4,5,479
5,6,472
6,7,387
7,8,408
8,9,357
9,10,297


,n_active_dates,n_service_ids
162,163,3
163,164,6
164,165,4
165,166,6
166,167,7
167,168,6
168,169,6
169,170,6
170,171,10
171,172,8


## Comment

The distribution confirms that the France GTFS calendar is very mixed.

Many `service_id` values are active only for a small number of dates. For example, there are many services that run on just 1, 2, 3, or a few days. At the same time, there is also a smaller group of services that run for a very long part of the feed, with values close to 180 days.

So the France GTFS calendar is not built around one simple regular pattern. It contains both many short-lived service ids and some long-running ones. This makes the calendar side more fragmented than in the earlier country cases.

## Link the GTFS calendar to trips

I now check how the `service_id` values from the France calendar are used in the trips table.

This helps me see whether the fragmented calendar structure is also reflected in the trip layer.

In [15]:
trip_counts_per_service = (
    trips.groupby("service_id")
    .size()
    .sort_values(ascending=False)
    .rename("n_trips")
    .reset_index()
)

display(trip_counts_per_service.head(20))

print("Unique service_id in trips:", trips["service_id"].nunique())
print("Unique service_id in calendar_dates:", calendar_dates["service_id"].nunique())

service_ids_only_in_trips = set(trips["service_id"]) - set(calendar_dates["service_id"])
service_ids_only_in_calendar = set(calendar_dates["service_id"]) - set(trips["service_id"])

print("service_id only in trips:", len(service_ids_only_in_trips))
print("service_id only in calendar_dates:", len(service_ids_only_in_calendar))

,service_id,n_trips
0,1605,1898
1,51,661
2,127,575
3,389,550
4,960,526
5,780,406
6,32,373
7,222,325
8,28,325
9,174,279


Unique service_id in trips: 12814
Unique service_id in calendar_dates: 12815
service_id only in trips: 0
service_id only in calendar_dates: 1


## Comment

The link between `trips` and `calendar_dates` is very consistent in the France GTFS feed.

The number of unique `service_id` values is almost identical in both tables: 12,814 in `trips` and 12,815 in `calendar_dates`. There are no `service_id` values that appear only in `trips`, and only one that appears only in `calendar_dates`.

This means that the calendar structure is well connected to the trip layer. So even though the France calendar is more fragmented than in earlier country cases, it still seems internally consistent on the GTFS side.

## Inspect the unmatched service_id

There is only one `service_id` that appears in `calendar_dates` but not in `trips`.

I inspect it directly to check whether this is only a small edge case or something more important.

In [16]:
only_in_calendar = list(service_ids_only_in_calendar)
print("service_id only in calendar_dates:", only_in_calendar)

calendar_dates_only_row = calendar_dates[
    calendar_dates["service_id"].isin(only_in_calendar)
].copy()

display(calendar_dates_only_row.head(20))
print("Rows for this unmatched service_id:", len(calendar_dates_only_row))

service_id only in calendar_dates: [0]


,service_id,date,exception_type
440867,0,20260426,1
440868,0,20260427,1
440869,0,20260428,1
440870,0,20260429,1
440871,0,20260430,1
440872,0,20260501,1
440873,0,20260502,1
440874,0,20260503,1
440875,0,20260504,1
440876,0,20260505,1


Rows for this unmatched service_id: 189


## Comment

The only unmatched `service_id` is `0`.

It is not a tiny edge case in terms of rows, because it appears on 189 dates, which is almost the full feed period. But it is still only one isolated `service_id`, and all the other service ids are shared between `trips` and `calendar_dates`.

So the overall GTFS structure still looks very consistent. The unmatched case should just be noted as a special exception that may represent an unused or placeholder service.

## Check the GTFS route types

As a final GTFS route check, I now look at the distribution of `route_type`.

This helps me see which transport modes are present in the France feed and gives a clearer picture of the line structure before closing the GTFS exploration.

In [17]:
route_type_labels = {
    0: "Tram",
    1: "Subway / Metro",
    2: "Rail",
    3: "Bus",
    4: "Ferry",
    5: "Cable Tram",
    6: "Aerial Lift",
    7: "Funicular",
    11: "Trolleybus",
    12: "Monorail"
}

print("route_type value counts:")
print(
    routes["route_type"]
    .value_counts(dropna=False)
    .sort_index()
    .rename(index=route_type_labels)
)

route_type value counts:
route_type
Tram      4
Rail    591
Bus     133
Name: count, dtype: int64


## Comment

The France GTFS feed is dominated by rail routes. Most route records are `Rail` (591), followed by a smaller number of `Bus` routes (133), while `Tram` appears only 4 times.

So the feed is mainly rail-based, but it is not purely rail.

## Summary: France GTFS

The France GTFS feed is large and structurally clear.

On the stop side, the feed is hierarchical. It contains station-level records (`location_type = 1`) and child stop records (`location_type = 0`) linked through `parent_station`. The station-level subset is clean, with no missing values in the main comparison fields.

On the line side, the route table is also clean. The main route fields are complete, but many visible line labels repeat across several technical route records. This suggests that `route_short_name` will likely be the more useful GTFS field later for the line-level comparison, rather than the raw `route_id`. The route types also show that the feed contains more than one transport mode.

On the calendar side, France differs clearly from Austria, Switzerland, and Luxembourg because the feed has no `calendar.txt` and uses only `calendar_dates`. Also, all rows use `exception_type = 1`, so `calendar_dates` seems to contain the real active service dates, not just small exceptions. The calendar structure is quite fragmented, with many short-lived service ids and some long-running ones. Still, the link between `trips` and `calendar_dates` is very consistent overall, with only one unmatched `service_id` (`0`) appearing only in `calendar_dates`.

# Part 2: France NeTEx Exploration

In [18]:
# data_dir was already defined in Part 1 (GTFS section) and is reused here
netex_zip_path = data_dir / "export-opendata-sncf-netex.zip"

print("NeTEx exists:", netex_zip_path.exists())
print("NeTEx path:", netex_zip_path)

NeTEx exists: True
NeTEx path: data/export-opendata-sncf-netex.zip


In [19]:
with zipfile.ZipFile(netex_zip_path, "r") as z:
    infos = z.infolist()

netex_files = pd.DataFrame([
    {
        "name": i.filename,
        "size_mb": round(i.file_size / (1024 * 1024), 2)
    }
    for i in infos
])

display(netex_files.sort_values("size_mb", ascending=False).head(30))

xml_files = netex_files[
    netex_files["name"].str.lower().str.endswith(".xml")
].sort_values("size_mb", ascending=False)

print("total files:", len(netex_files))
print("xml files:", len(xml_files))

display(xml_files.head(30))

,name,size_mb
0,sncf_netexfr_20260426_2233.xml,520.48
1,trf2netex.log,72.79


total files: 2
xml files: 1


,name,size_mb
0,sncf_netexfr_20260426_2233.xml,520.48


## Comment

The France NeTEx ZIP is very simple.

It contains only one main XML file and one log file. So unlike Luxembourg, the France NeTEx data is not split across many XML files.

This means the next steps will focus on one large XML file.

## Inspect the basic structure of the France NeTEx XML

The France NeTEx ZIP contains one main XML file, so the next step is to inspect its basic structure.

At this point, I only check the root element and namespaces to confirm that the file looks like a normal Netex export before going deeper into stops, lines etc.

In [20]:
xml_name = xml_files.iloc[0]["name"]

with zipfile.ZipFile(netex_zip_path, "r") as z:
    with z.open(xml_name) as f:
        context = ET.iterparse(f, events=("start", "start-ns"))
        
        namespaces = {}
        root = None
        
        for event, elem in context:
            if event == "start-ns":
                prefix, uri = elem
                namespaces[prefix if prefix else "default"] = uri
            elif event == "start":
                root = elem
                break

print("XML file:", xml_name)
print("Root tag:", root.tag if root is not None else None)
print("\nNamespaces:")
for k, v in namespaces.items():
    print(f"{k}: {v}")

XML file: sncf_netexfr_20260426_2233.xml
Root tag: {http://www.netex.org.uk/netex}PublicationDelivery

Namespaces:
gml: http://www.opengis.net/gml/3.2
xsi: http://www.w3.org/2001/XMLSchema-instance
siri: http://www.siri.org.uk/siri
default: http://www.netex.org.uk/netex


## Inspect the first level of the France NeTEx structure

Now I inspect the first level under `PublicationDelivery` (the root container that holds all the published transport data).

This helps me see which main blocks are present in the file and where the later stop, line, and calendar-related objects are likely stored.

In [21]:
# Check the file size first before parsing

with zipfile.ZipFile(netex_zip_path, "r") as z:
    for info in z.infolist():
        print(f"{info.filename}: {info.file_size / (1024*1024):.1f} MB")

sncf_netexfr_20260426_2233.xml: 520.5 MB
trf2netex.log: 72.8 MB


## Peeking at the raw XML without parsing

Since the NeTEx file is 520 MB, loading it fully into memory with a standard XML parser would freeze the kernel. Instead, I read only the first 5,000 bytes of the raw file as plain text. This is enough to see the top-level structure, the namespace, and the first few elements without touching the rest of the file. No parsing is needed at all for this step.

In [22]:
# Just peek at the raw XML text (no parsing needed)
with zipfile.ZipFile(netex_zip_path, "r") as z:
    with z.open(xml_name) as f:
        head = b""
        while len(head) < 5000:
            chunk = f.read(1024)
            if not chunk:
                break
            head += chunk

print(head.decode("utf-8", errors="replace"))

<?xml version="1.0" encoding="utf-8"?>
<PublicationDelivery xmlns:gml="http://www.opengis.net/gml/3.2" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:siri="http://www.siri.org.uk/siri" version="1.09" xmlns="http://www.netex.org.uk/netex" xsi:schemaLocation="http://www.netex.org.uk/netex">
  <PublicationTimestamp>2026-04-26T22:33:51</PublicationTimestamp>
  <ParticipantRef>HaCon</ParticipantRef>
  <Description lang="fr">NeTEx Export, Revision: 2.23 </Description>
  <dataObjects>
    <CompositeFrame id="FR:1:CompositeFrame:qv1" version="HRDF">
      <ValidBetween>
        <FromDate>2026-04-26T00:00:00</FromDate>
        <ToDate>2026-09-23T23:59:59</ToDate>
      </ValidBetween>
      <FrameDefaults>
        <DefaultLocale>
          <TimeZoneOffset>+1</TimeZoneOffset>
          <SummerTimeZoneOffset>+2</SummerTimeZoneOffset>
          <DefaultLanguage>fr</DefaultLanguage>
        </DefaultLocale>
      </FrameDefaults>
      <frames>
        <GeneralFrame id="FR:1:GeneralFra

## Comment

The root element is `PublicationDelivery`, which is the standard NeTEx wrapper. Inside it I can see one `CompositeFrame` covering **April 26 to September 23, 2026**.

The most important observation is that the first elements visible in the `<members>` block are `JourneyPart` objects, individual segments of train journeys with departure and arrival stops and times. This means the file jumps straight into trip-level detail, which is very different from Luxembourg where `StopPlace` and `Line` elements appeared near the top.


## 2.1 Extracting Stations from the NeTEx XML File

Since the France NeTEx feed is a single 520 MB file, I cannot load it directly into Python because it would crash the kernel. I therefore split the extraction into two steps: first unpack the XML file from the ZIP archive onto disk (in this case the university server's storage), and then use a command line tool to pull out only the station-related lines without reading the whole file into memory.

### Step 1: Extract the XML file to disk

First unpack the compressed ZIP archive and save the XML file directly to disk. 

In [23]:
# Extract the XML to disk to not read through the ZIP stream
extract_dir = Path("data/netex_extracted")
extract_dir.mkdir(exist_ok=True)

print("Extracting XML file to disk...")
with zipfile.ZipFile(netex_zip_path, "r") as z:
    z.extract(xml_name, extract_dir)

extracted_path = extract_dir / xml_name
print(f"Done. File size: {extracted_path.stat().st_size / (1024*1024):.1f} MB")

Extracting XML file to disk...
Done. File size: 520.5 MB


### Step 2: Pull out only the StopPlace lines

Instead of loading the full 520 MB file into Python, I use `grep` which is a command line tool that scans a file line by line and extracts only the lines that contain a specific word. Here I search for the word `StopPlace`, which reduces the 520 MB file down to just **1.4 MB**. I then preview the first few lines to check what the station data looks like.

In [24]:
# Use grep to extract only lines containing StopPlace tags
output_path = extract_dir / "stopplaces_only.txt"

subprocess.run(
    f"grep -i 'StopPlace' '{extracted_path}' > '{output_path}'",
    shell=True
)

print(f"Done. Output size: {output_path.stat().st_size / (1024*1024):.1f} MB")

# Preview first few lines
with open(output_path, "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 10:
            break

Done. Output size: 1.4 MB
<TypeOfPlace id="multimodalStopPlace" version="any">
<Name>multimodalStopPlace</Name>
<TypeOfPlace id="monomodalStopPlace" version="any">
<Name>monomodalStopPlace</Name>
<stopPlaces>
<StopPlace id="FR::LMO:72425e40-3a69-11e9-8417-bb1d8a705241:" created="2019-02-27T00:00:00.000" changed="2025-09-17T00:00:00.000" modification="revise" version="1">
<TypeOfPlaceRef ref="monomodalStopPlace" version="any" />
<StopPlaceType>railStation</StopPlaceType>
</StopPlace>
<StopPlace id="FR::LMO:c249c6f0-fa0b-11e9-b755-b1228bd14244:" created="2019-10-29T00:00:00.000" changed="2025-09-17T00:00:00.000" modification="revise" version="1">
<TypeOfPlaceRef ref="monomodalStopPlace" version="any" />
<StopPlaceType>railStation</StopPlaceType>


## Output

The `grep` extraction worked and reduced the file to **1.4 MB**. The preview shows station entries of type `railStation`, which is expected for the SNCF feed. However, there are **no coordinates visible** in these lines.

This is not necessarily because coordinates are missing, it is because `grep` only extracts lines that contain the exact word "StopPlace". Since XML spreads the content of each element across multiple lines, any coordinate tags sitting inside the `<StopPlace>` block but on their own separate lines would not be picked up. To check whether coordinates are actually present, I need to extract the full `<StopPlace>` blocks rather than just the matching lines.

## Extracting full StopPlace blocks with `sed`

Since `grep` only returned individual lines and missed the coordinates, I use a different command line tool called `sed` instead. While `grep` looks for single lines containing a word, `sed` can extract everything between a start and an end pattern, in this case the full block from `<StopPlace` to `</StopPlace>`, including all the nested elements in between such as names, coordinates, and addresses.

This gives a much more complete picture of each station entry. The output is saved as a new file on the server's storage and then previewed to check what is inside.

In [25]:
# Use sed to extract full StopPlace blocks (opening to closing tag)
stopplace_blocks_path = extract_dir / "stopplace_blocks.txt"

subprocess.run(
    f"sed -n '/<StopPlace /,/<\\/StopPlace>/p' '{extracted_path}' > '{stopplace_blocks_path}'",
    shell=True
)

print(f"Done. Output size: {stopplace_blocks_path.stat().st_size / (1024*1024):.1f} MB")

# Preview first 30 lines
with open(stopplace_blocks_path, "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 30:
            break

Done. Output size: 6.0 MB
<StopPlace id="FR::LMO:72425e40-3a69-11e9-8417-bb1d8a705241:" created="2019-02-27T00:00:00.000" changed="2025-09-17T00:00:00.000" modification="revise" version="1">
<ValidBetween>
<FromDate>2026-04-26T00:00:00</FromDate>
<ToDate>2026-09-23T23:59:59</ToDate>
</ValidBetween>
<Name lang="fr">FIGUERES-VILAFANT</Name>
<ShortName lang="fr">FIGUERES-VILAFANT</ShortName>
<PrivateCode></PrivateCode>
<Centroid>
<Location srsName="EPSG::4326">
<Longitude>2.943028</Longitude>
<Latitude>42.264581</Latitude>
</Location>
</Centroid>
<placeTypes>
<TypeOfPlaceRef ref="monomodalStopPlace" version="any" />
</placeTypes>
<PostalAddress id="FR:PostalAddress::72425e40-3a69-11e9-8417-bb1d8a705241" version="1">
<CountryRef>ES</CountryRef>
<HouseNumber>S/N</HouseNumber>
<Street>Camí vell d'Avinyonet</Street>
<Town>Vilafant</Town>
<PostCode>17740</PostCode>
<PostalRegion></PostalRegion>
</PostalAddress>
<Locale>
<TimeZoneOffset>+1</TimeZoneOffset>
<TimeZone>CET</TimeZone>
<SummerTimeZo

## Comment

The full `StopPlace` block is now visible and contains all the information we need. A few key observations:

**Coordinates are present** and they are stored inside a `<Centroid>` → `<Location>` block, slightly differently from Luxembourg but perfectly extractable. For this first station I can see`<Longitude>2.943028</Longitude>` and `<Latitude>42.264581</Latitude>`.

**Each station contains:**
- `<Name>` — the full station name (`FIGUERES-VILAFANT`)
- `<ShortName>` — a shorter version of the name
- `<ValidBetween>` — the validity period (April 26 to September 23, 2026)
- `<TransportMode>` — `rail`, as expected for SNCF
- `<PostalAddress>` — full address including street, town, postcode and country

**One interesting detail:** the `<CountryRef>` for this station is `ES` (Spain), not `FR` (France). This makes sense because SNCF operates some cross-border international routes, so stations just across the border in Spain or other neighbouring countries can appear in the feed.


## Parsing the StopPlace blocks into a DataFrame

The original 520 MB NeTEx file is a complete XML document with one root element wrapping everything inside it. When I used `sed` to extract only the `<StopPlace>` blocks, I pulled those blocks out into a new file, but that new file has no root element wrapping them together. XML requires exactly one root element, so before parsing I wrap the extracted content in an artificial `<root>` tag to make it valid again.

Once the XML is valid, I loop through every `<StopPlace>` element and extract four fields: the station ID, the name, the latitude, and the longitude. The coordinates are stored inside a `<Centroid>` → `<Location>` block, so I search for them using a path that looks anywhere inside the element. The result is stored in a DataFrame called `netex_stops_fr`.

In [26]:
# Wrap the extracted blocks in a root element to make it valid XML
with open(stopplace_blocks_path, "r", encoding="utf-8") as f:
    content = f.read()

# Wrap with a root tag and the NeTEx namespace
wrapped = f'<root xmlns="http://www.netex.org.uk/netex">{content}</root>'

# Parse the wrapped XML
root = ET.fromstring(wrapped)

netex_tag = "{http://www.netex.org.uk/netex}"

all_stopplace_rows = []
for sp in root.findall(f"{netex_tag}StopPlace"):
    lat_elem  = sp.find(f".//{netex_tag}Latitude")
    lon_elem  = sp.find(f".//{netex_tag}Longitude")
    name_elem = sp.find(f"{netex_tag}Name")

    all_stopplace_rows.append({
        "stopplace_id":   sp.get("id"),
        "stopplace_name": name_elem.text.strip() if name_elem is not None and name_elem.text else None,
        "latitude":       lat_elem.text.strip() if lat_elem is not None and lat_elem.text else None,
        "longitude":      lon_elem.text.strip() if lon_elem is not None and lon_elem.text else None,
    })

netex_stops_fr = pd.DataFrame(all_stopplace_rows)

print(f"Total StopPlace rows: {len(netex_stops_fr):,}")
print(f"Missing latitude:  {netex_stops_fr['latitude'].isna().sum()}")
print(f"Missing longitude: {netex_stops_fr['longitude'].isna().sum()}")
display(netex_stops_fr.head(10))

Total StopPlace rows: 3,506
Missing latitude:  0
Missing longitude: 0


,stopplace_id,stopplace_name,latitude,longitude
0,FR::LMO:72425e40-3a69-11e9-8417-bb1d8a705241:,FIGUERES-VILAFANT,42.264581,2.943028
1,FR::LMO:c249c6f0-fa0b-11e9-b755-b1228bd14244:,Barcelone-Sants,41.378961,2.139834
2,FR::LMO:7247b570-3a69-11e9-8417-bb1d8a705241:,GIRONA,41.979371,2.816957
3,FR::LMO:c24903a0-fa0b-11e9-b755-b1228bd14244:,Portbou,42.424701,3.158034
4,FR::LMO:71425360-3a69-11e9-8417-bb1d8a705241:,Augsburg Hbf,48.365400,10.886000
5,FR::LMO:71a3fb60-3a69-11e9-8417-bb1d8a705241:,Berlin-Gesundbrunnen,52.548800,13.391000
6,FR::LMO:71bb05d0-3a69-11e9-8417-bb1d8a705241:,Frankfurt (Main) Sued,50.099200,8.686100
7,FR::LMO:71ba90a0-3a69-11e9-8417-bb1d8a705241:,Francfort sur le Main,50.107300,8.662400
8,FR::LMO:71cdf190-3a69-11e9-8417-bb1d8a705241:,Mannheim Hbf,49.479400,8.469600
9,FR::LMO:71d21040-3a69-11e9-8417-bb1d8a705241:,Heidelberg Hbf,49.403500,8.675400


## Output

The parsing worked cleanly: **3,506 StopPlace rows** with **zero missing coordinates** on both latitude and longitude were extracted.

Looking at the preview, a few interesting things stand out: 

- the first stations visible are not in France at all: there are stations in **Spain** (Figueres-Vilafant, Barcelone-Sants, Girona), **Germany** (Augsburg Hbf, Berlin-Gesundbrunnen, Frankfurt, Mannheim, Heidelberg), and other neighbouring countries. This confirms that the SNCF NeTEx feed includes all stations served by international routes, not just French ones

- the `stopplace_id` follows the format `FR::LMO:uuid:`, which is very different from Luxembourg where the ID contained a readable numeric code that matched the GTFS `stop_id` directly. This means **ID-based matching with GTFS will not work the same way here**, and I will likely need to rely on **coordinate-based matching** or **name-based matching** instead

## Inspecting the NeTEx StopPlace table


In [27]:
print("Shape:", netex_stops_fr.shape)
print("\nMissingness:")
print(netex_stops_fr.isna().sum())
print("\nSample stopplace_id values:")
print(netex_stops_fr["stopplace_id"].head(10).tolist())
print("\nUnique StopPlaces:", netex_stops_fr["stopplace_id"].nunique())

Shape: (3506, 4)

Missingness:
stopplace_id      0
stopplace_name    0
latitude          0
longitude         0
dtype: int64

Sample stopplace_id values:
['FR::LMO:72425e40-3a69-11e9-8417-bb1d8a705241:', 'FR::LMO:c249c6f0-fa0b-11e9-b755-b1228bd14244:', 'FR::LMO:7247b570-3a69-11e9-8417-bb1d8a705241:', 'FR::LMO:c24903a0-fa0b-11e9-b755-b1228bd14244:', 'FR::LMO:71425360-3a69-11e9-8417-bb1d8a705241:', 'FR::LMO:71a3fb60-3a69-11e9-8417-bb1d8a705241:', 'FR::LMO:71bb05d0-3a69-11e9-8417-bb1d8a705241:', 'FR::LMO:71ba90a0-3a69-11e9-8417-bb1d8a705241:', 'FR::LMO:71cdf190-3a69-11e9-8417-bb1d8a705241:', 'FR::LMO:71d21040-3a69-11e9-8417-bb1d8a705241:']

Unique StopPlaces: 3505


## Comment

The table has **3,506 rows and 4 columns**, with **zero missing values** across all fields.

There is one duplicate: 3,506 rows but only **3,505 unique StopPlace IDs**.

## 2.2 Extracting Lines from the NeTEx XML File

I use the same `sed` approach as for the stations. Instead of loading the full 520 MB file into Python, I scan it at the operating system level and extract only the blocks that contain `<Line>` elements. This gives a much smaller file that Python can then parse normally. I also preview the first few lines to check what fields are available inside each `<Line>` block before writing the full extraction code.

In [28]:
# Use sed to extract full Line blocks
line_blocks_path = extract_dir / "line_blocks.txt"

subprocess.run(
    f"sed -n '/<Line /,/<\\/Line>/p' '{extracted_path}' > '{line_blocks_path}'",
    shell=True
)

print(f"Done. Output size: {line_blocks_path.stat().st_size / (1024*1024):.1f} MB")

# Preview first 30 lines
with open(line_blocks_path, "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 30:
            break

Done. Output size: 2.9 MB
<Line id="FR:Line::00F2577A-6A87-42E0-95F3-07351E4BC2F6:" version="any" responsibilitySetRef="FR:ResponsibilitySet::d2ecc4f8-b4c7-4900-ae81-e4f60290b2d4:">
<BrandingRef>FLUO</BrandingRef>
<Name>Bening - Sarreguemines</Name>
<Description></Description>
<TransportMode>coach</TransportMode>
<TransportSubmode>
<CoachSubmode>regionalCoach</CoachSubmode>
</TransportSubmode>
<PublicCode>P53</PublicCode>
<OperatorRef ref="FR:Operator::07B82396-04B1-48F0-84BF-36C7F1B81442:" version="any"/>
<routes>
<RouteRef ref="FR:Route::13dda818-dd75-49d4-ae8b-4614623c7575:"/>
<RouteRef ref="FR:Route::158f6e06-16e5-4f1d-9d12-b841b58b8124:"/>
<RouteRef ref="FR:Route::1a351c6e-2c16-46d6-9132-d7148d6a4dd6:"/>
<RouteRef ref="FR:Route::2204c9d9-fa3c-4978-92a7-0326c31e2d56:"/>
<RouteRef ref="FR:Route::35fb349a-3dd6-4d89-a628-8d0c41d3623e:"/>
<RouteRef ref="FR:Route::437b4d6e-2b62-4a3c-ab48-071d37de3156:"/>
<RouteRef ref="FR:Route::4c8f830a-73c3-4cb2-8c8d-d2f7497dd102:"/>
<RouteRef ref="FR

## Comment

The `sed` extraction worked and reduced the data to **2.9 MB**. Each `<Line>` block contains the following fields:

- `<Name>` — the full line name (e.g. `Bening – Sarreguemines`)
- `<PublicCode>` — the short public code (e.g. `P53`), which is the equivalent of `route_short_name` in GTFS
- `<TransportMode>` — the main transport mode (e.g. `coach`)
- `<TransportSubmode>` — a more specific submode (e.g. `regionalCoach`)
- `<BrandingRef>` — a reference to the operator branding (e.g. `FLU0`)
- `<routes>` — a list of route references that belong to this line
- `<Presentation>` — colour information for the line

## Parsing the Line blocks into a DataFrame

The 2.9 MB file extracted by `sed` is small enough to parse normally. I apply the same approach as for the stations. I wrap the extracted blocks in a `<root>` tag to make it a valid XML document, then loop through every `<Line>` element and extract five fields: the line ID, the full name, the public code, the transport mode, and the submode. For the submode I check three possible tags (`CoachSubmode`, `RailSubmode`, `BusSubmode`) since different lines use different submode elements depending on their transport type. The result is stored in a DataFrame called `netex_lines_fr`.

In [29]:
# Read and wrap the extracted Line blocks
with open(line_blocks_path, "r", encoding="utf-8") as f:
    content = f.read()

wrapped = f'<root xmlns="http://www.netex.org.uk/netex">{content}</root>'
root = ET.fromstring(wrapped)
netex_tag = "{http://www.netex.org.uk/netex}"

all_line_rows = []
for line in root.findall(f"{netex_tag}Line"):
    name_elem    = line.find(f"{netex_tag}Name")
    code_elem    = line.find(f"{netex_tag}PublicCode")
    mode_elem    = line.find(f"{netex_tag}TransportMode")

    # Note: ElementTree elements with no child tags are falsy in a boolean
    # context regardless of their text content, so "elem1 or elem2 or elem3"
    # silently skips a real match. Check "is not None" explicitly instead.
    coach_elem = line.find(f".//{netex_tag}CoachSubmode")
    rail_elem  = line.find(f".//{netex_tag}RailSubmode")
    bus_elem   = line.find(f".//{netex_tag}BusSubmode")
    if coach_elem is not None:
        submode_elem = coach_elem
    elif rail_elem is not None:
        submode_elem = rail_elem
    else:
        submode_elem = bus_elem

    all_line_rows.append({
        "line_id":        line.get("id"),
        "line_name":      name_elem.text.strip() if name_elem is not None and name_elem.text else None,
        "public_code":    code_elem.text.strip() if code_elem is not None and code_elem.text else None,
        "transport_mode": mode_elem.text.strip() if mode_elem is not None and mode_elem.text else None,
        "submode":        submode_elem.text.strip() if submode_elem is not None and submode_elem.text else None,
    })

netex_lines_fr = pd.DataFrame(all_line_rows)

print(f"Total Line rows:   {len(netex_lines_fr):,}")
print(f"Unique line_id:    {netex_lines_fr['line_id'].nunique():,}")
print(f"\nMissingness:")
print(netex_lines_fr.isna().sum())
print(f"\nTransport mode distribution:")
print(netex_lines_fr["transport_mode"].value_counts(dropna=False))

display(netex_lines_fr.head(10))

Total Line rows:   728
Unique line_id:    728

Missingness:
line_id             0
line_name           0
public_code         0
transport_mode      0
submode           143
dtype: int64

Transport mode distribution:
transport_mode
rail       577
coach       87
unknown     64
Name: count, dtype: int64


,line_id,line_name,public_code,transport_mode,submode
0,FR:Line::00F2577A-6A87-42E0-95F3-07351E4BC2F6:,Bening - Sarreguemines,P53,coach,regionalCoach
1,FR:Line::00F7208C-CEBC-4521-A792-6EC3ABB65811:,Saint-Étienne - Roanne,C30,rail,regionalRail
2,FR:Line::00e52513-07a0-4ce1-8b13-97cab39bdef8:,Ligne S3 BreizhGo,S3,rail,touristRailway
3,FR:Line::0128E1D5-9183-4D58-B1CF-F5AA5A64A037:,Marseille - Toulon - Hyeres,C6,rail,regionalRail
4,FR:Line::0202671B-7107-429E-A37B-473C55E0254C:,Montpellier Saint-Roch - Avignon Centre,C8,rail,regionalRail
5,FR:Line::022B77D9-D121-4DCB-B808-FB2F7931866B:,Nantes Cholet Angers Saumur,C20,rail,regionalRail
6,FR:Line::02534A5F-903C-454E-A24F-E6E2E23B3CBF:,Ambérieu - Mâcon,P13,rail,regionalRail
7,FR:Line::0312E125-B6BC-404C-A7D1-C1600D4CACAF:,Toulouse Matabiau - Brive La Gaillarde,K1,rail,regionalRail
8,FR:Line::03398D75-5B60-4588-BC05-9985E599BA20:,Ussel - Bort Les Orgues,R16,coach,regionalCoach
9,FR:Line::0359c353-2f81-4c49-a269-bc287ff237e0:,22. Limoges - Uzerche - Brive,L22,rail,regionalRail


## Comment

The extraction worked cleanly: **728 unique Lines** with zero missing values on `line_id`, `line_name`, `public_code` and `transport_mode`. The `submode` field is missing for 143 of the 728 rows. These are lines whose `<Line>` block does not carry a `CoachSubmode`/`RailSubmode`/`BusSubmode` tag directly (submode is optional at the line level in this feed). Where present, `submode` gives a finer-grained classification such as `regionalRail`, `regionalCoach`, `interregionalRail` or `highSpeedRail`. This is not a problem for the comparison since `transport_mode` is already available and complete for all 728 rows.

The transport mode distribution shows three modes:
- **rail** — 577 lines, the dominant mode as expected for SNCF
- **coach** — 87 lines, regional coach services
- **unknown** — 64 lines, where the mode was not specified in the feed

The number of lines matches the GTFS routes count of **728** exactly, the same pattern observed earlier with stations. 

The `public_code` field (e.g. `P53`, `C30`, `S3`) is the NeTEx equivalent of `route_short_name` in GTFS, and will be the key field for the line-level comparison in Part 3.


## Checking Public Code Uniqueness

Since `public_code` is the field I plan to use to match NeTEx lines to GTFS routes, I first check whether it is unique. If multiple lines share the same public code, a simple join on that field would produce incorrect matches.

In [30]:
# Check public_code uniqueness (same code can refer to multiple lines)
print("Unique line_id:    ", netex_lines_fr["line_id"].nunique())
print("Unique public_code:", netex_lines_fr["public_code"].nunique())

# Show which public codes appear more than once
public_code_counts = (
    netex_lines_fr["public_code"]
    .value_counts()
    .reset_index(name="n_lines")
    .rename(columns={"index": "public_code"})
)

print("\nPublic codes appearing more than once:", (public_code_counts["n_lines"] > 1).sum())
display(public_code_counts[public_code_counts["n_lines"] > 1].head(10))

Unique line_id:     728
Unique public_code: 427

Public codes appearing more than once: 107


,public_code,n_lines
0,INCONNU,64
1,P14,7
2,P4,7
3,P3,6
4,P11,6
5,P10,6
6,C1,6
7,P2,6
8,C2,6
9,C10,6


## Comment
There are **728 unique line IDs** but only **427 unique public codes**, meaning 107 public codes appear more than once. The most striking case is `INCONNU` (French for "unknown") which appears 64 times. These are the same 64 lines that had `transport_mode = unknown` earlier, confirming that a group of lines in the feed have no proper identifier assigned.

The remaining duplicates (P14, P4, K2, etc.) likely represent the same route code used in different regions of France.

This confirms that matching on `public_code` alone will not be reliable. For the line-level comparison in Part 3 I will need to take this into account.

## 2.3 Extracting Calendar from the NeTEx XML File

Before extracting the calendar elements, I first need to find out which calendar-related tags are actually present in the France NeTEx file. Unlike Luxembourg, France has no `calendar.txt` in GTFS  only `calendar_dates`. So the NeTEx calendar structure might also be different. I use `grep` to scan the full file and list all unique XML tags, which tells me exactly what calendar elements are available before I start extracting.

In [31]:
# Use grep to find which calendar-related tags exist in the file
subprocess.run(
    f"grep -o '<[A-Za-z]*>' '{extracted_path}' | sort | uniq -c | sort -rn | head -50 > '{extract_dir}/tags.txt'",
    shell=True
)

with open(extract_dir / "tags.txt", "r") as f:
    print(f.read())

427986 <TimetabledPassingTime>
427986 <DepartureTime>
378511 <ArrivalTime>
134629 <Distance>
71421 <ToDate>
71421 <FromDate>
59994 <ValidBetween>
54083 <TransportMode>
50060 <TransportSubmode>
49476 <trainNumbers>
49476 <dayTypes>
49475 <passingTimes>
49475 <ServiceAlteration>
49475 <ForAdvertisement>
48772 <facilities>
48772 <FareClasses>
48772 <AccommodationFacilityList>
44246 <RailSubmode>
40013 <pointsInSequence>
37727 <GroupBookingFacility>
31666 <LuggageCarriageFacilityList>
30724 <DirectionType>
15325 <StartTime>
15325 <EndTime>
14544 <Longitude>
14544 <Latitude>
11427 <ValidDayBits>
10843 <ServiceReservationFacilityList>
10110 <PassengerCommsFacilityList>
9703 <FamilyFacilityList>
8947 <ServiceJourneyPatternType>
8576 <CateringFacilityList>
6291 <PrivateCode>
6272 <CountryRef>
5814 <CoachSubmode>
4239 <PublicCode>
3507 <TimeZoneOffset>
3507 <SummerTimeZoneOffset>
3507 <DefaultLanguage>
3506 <placeTypes>
3506 <TimeZone>
3506 <StopPlaceType>
3506 <Locale>
3506 <Centroid>
3490 <To

## Comment

The tag list confirms that the France NeTEx file does contain calendar-related elements. The most relevant ones are:

- `<ValidBetween>` (59,994 occurrences) — defines the validity period with `<FromDate>` and `<ToDate>`
- `<dayTypes>` (49,476 occurrences) — references to the DayType elements
- `<ValidDayBits>` (11,427 occurrences) — the binary bit string encoding which days a service runs, the same format used in Luxembourg

This means the calendar structure is similar to Luxembourg. `DayType` and `ValidDayBits` are both present. However the counts are much larger, reflecting the fact that this is a national feed covering the entire SNCF network with tens of thousands of individual services.

Also worth noting: `<RailSubmode>` appears 44,246 times and `<CoachSubmode>` 5,814 times across the whole file, far more than the 728 `<Line>` blocks extracted earlier. Most of these occurrences sit at the individual service-journey level rather than the line level, but many `<Line>` blocks do also carry their own `CoachSubmode`/`RailSubmode` tag (see the corrected extraction below).

The next step is to extract the `DayType` and `UicOperatingPeriod` blocks using the same `sed` approach.


## Extracting UicOperatingPeriod blocks

I use the same `sed` approach to extract all `UicOperatingPeriod` blocks from the 520 MB file. These blocks contain the actual calendar data, specifically the `ValidDayBits` string that encodes which days each service runs, along with the `FromDate` and `ToDate` defining the validity window.

In [32]:
# Extract UicOperatingPeriod blocks
operatingperiod_blocks_path = extract_dir / "operatingperiod_blocks.txt"

subprocess.run(
    f"sed -n '/<UicOperatingPeriod /,/<\\/UicOperatingPeriod>/p' '{extracted_path}' > '{operatingperiod_blocks_path}'",
    shell=True
)

print(f"Done. Output size: {operatingperiod_blocks_path.stat().st_size / (1024*1024):.1f} MB")

# Preview first 30 lines
with open(operatingperiod_blocks_path, "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 30:
            break

Done. Output size: 4.5 MB
<UicOperatingPeriod id="FR:OperatingPeriod:1" version="any">
<FromDate>2026-04-26T00:00:00</FromDate>
<ToDate>2026-09-23T23:59:59</ToDate>
<ValidDayBits>1111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111</ValidDayBits>
</UicOperatingPeriod>
<UicOperatingPeriod id="FR:OperatingPeriod:2" version="any">
<FromDate>2026-04-26T00:00:00</FromDate>
<ToDate>2026-09-23T23:59:59</ToDate>
<ValidDayBits>0000000000000000000000000000000000000000000000000000000000000000000000011111001011100111110011111001111100111110011111001111100000000000000000000000000</ValidDayBits>
</UicOperatingPeriod>
<UicOperatingPeriod id="FR:OperatingPeriod:3" version="any">
<FromDate>2026-04-26T00:00:00</FromDate>
<ToDate>2026-09-23T23:59:59</ToDate>
<ValidDayBits>00000000000000000000000000000000000000000000000000000000000000000000000000000010000000000000000000000000000000000000000000000000000000000

## Comment

The extraction produced a **4.5 MB** file which is small enough to parse normally. Each `UicOperatingPeriod` block contains exactly three fields:

- `<FromDate>` and `<ToDate>` - all periods share the same validity window: **April 26 to September 23, 2026**, which is the full SNCF feed window
    
- `<ValidDayBits>` - a long binary string where each `1` means the service runs on that day and `0` means it does not. The bit strings are much longer than in Luxembourg because the France feed covers a much longer period

The IDs follow a simple sequential pattern (`FR:OperatingPeriod:1`, `FR:OperatingPeriod:2`, etc.), which is different from Luxembourg's more complex ID format.


## Parsing the UicOperatingPeriod blocks into a DataFrame

The 4.5 MB extracted file is small enough to parse normally. I apply the same approach as for stations and lines  wrapping the blocks in a `<root>` tag to make it valid XML, then looping through every `UicOperatingPeriod` element and extracting four fields: the period ID, the start date, the end date, and the `ValidDayBits` string. I also slice the dates to keep only the first 10 characters (e.g. `2026-04-26`) and strip the time component.

In [33]:
# Read and wrap the extracted UicOperatingPeriod blocks
with open(operatingperiod_blocks_path, "r", encoding="utf-8") as f:
    content = f.read()

wrapped = f'<root xmlns="http://www.netex.org.uk/netex">{content}</root>'
root = ET.fromstring(wrapped)
netex_tag = "{http://www.netex.org.uk/netex}"

all_operatingperiod_rows = []
for op in root.findall(f"{netex_tag}UicOperatingPeriod"):
    from_elem = op.find(f"{netex_tag}FromDate")
    to_elem   = op.find(f"{netex_tag}ToDate")
    bits_elem = op.find(f"{netex_tag}ValidDayBits")

    all_operatingperiod_rows.append({
        "operating_period_id": op.get("id"),
        "from_date":           from_elem.text.strip()[:10] if from_elem is not None and from_elem.text else None,
        "to_date":             to_elem.text.strip()[:10] if to_elem is not None and to_elem.text else None,
        "valid_day_bits":      bits_elem.text.strip() if bits_elem is not None and bits_elem.text else None,
    })

netex_operatingperiods_fr = pd.DataFrame(all_operatingperiod_rows)

print(f"Total UicOperatingPeriod rows: {len(netex_operatingperiods_fr):,}")
print(f"\nMissingness:")
print(netex_operatingperiods_fr.isna().sum())
print(f"\nDate range:")
print("From:", netex_operatingperiods_fr["from_date"].min())
print("To:  ", netex_operatingperiods_fr["to_date"].max())

display(netex_operatingperiods_fr.head(10))

Total UicOperatingPeriod rows: 11,427

Missingness:
operating_period_id    0
from_date              0
to_date                0
valid_day_bits         0
dtype: int64

Date range:
From: 2026-04-26
To:   2026-09-23


,operating_period_id,from_date,to_date,valid_day_bits
0,FR:OperatingPeriod:1,2026-04-26,2026-09-23,1111111111111111111111111111111111111111111111...
1,FR:OperatingPeriod:2,2026-04-26,2026-09-23,0000000000000000000000000000000000000000000000...
2,FR:OperatingPeriod:3,2026-04-26,2026-09-23,0000000000000000000000000000000000000000000000...
3,FR:OperatingPeriod:4,2026-04-26,2026-09-23,0000000000000000000000000000000000000000000000...
4,FR:OperatingPeriod:5,2026-04-26,2026-09-23,1000111100011110011111000011110001110000111000...
5,FR:OperatingPeriod:6,2026-04-26,2026-09-23,0111100011110001110000111110001111001111100111...
6,FR:OperatingPeriod:7,2026-04-26,2026-09-23,0100000010000001000000100000001000001000000100...
7,FR:OperatingPeriod:8,2026-04-26,2026-09-23,0000000000000000010000000000000000000000000000...
8,FR:OperatingPeriod:9,2026-04-26,2026-09-23,0000000000000000000000001000000000000000000000...
9,FR:OperatingPeriod:10,2026-04-26,2026-09-23,0111100011110001100000110110001111001111100111...


## Output

The extraction produced **11,427 unique operating periods** with zero missing values across all fields. This is a much larger number than Luxembourg (2,042), reflecting the fact that France covers a much larger and more complex network.

All periods share the same date range: **April 26 to September 23, 2026**, which is the full SNCF feed validity window. 
    
Looking at the preview, the bit strings already show clear patterns: `FR:OperatingPeriod:1` has all `1`s (runs every day), while others have recurring patterns that suggest weekly schedules (e.g. Monday to Friday, or weekends only).


## Extracting DayType blocks

I use the same `sed` approach to extract all `DayType` blocks from the 520 MB file. In NeTEx, `DayType` is a named calendar rule that defines which days a service runs. It is linked to a `UicOperatingPeriod` through a `DayTypeAssignment`, which together encode the full service validity.

In [34]:
# Extract DayType blocks using sed
daytype_blocks_path = extract_dir / "daytype_blocks.txt"

subprocess.run(
    f"sed -n '/<DayType /,/<\\/DayType>/p' '{extracted_path}' > '{daytype_blocks_path}'",
    shell=True
)

print(f"Done. Output size: {daytype_blocks_path.stat().st_size / (1024*1024):.1f} MB")

# Preview first 30 lines
with open(daytype_blocks_path, "r") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 30:
            break

Done. Output size: 238.3 MB
<DayType id="FR:DayType:1:" version="any"/>
<DayType id="FR:DayType:2:" version="any"/>
<DayType id="FR:DayType:3:" version="any"/>
<DayType id="FR:DayType:4:" version="any"/>
<DayType id="FR:DayType:5:" version="any"/>
<DayType id="FR:DayType:6:" version="any"/>
<DayType id="FR:DayType:7:" version="any"/>
<DayType id="FR:DayType:8:" version="any"/>
<DayType id="FR:DayType:9:" version="any"/>
<DayType id="FR:DayType:10:" version="any"/>
<DayType id="FR:DayType:11:" version="any"/>
<DayType id="FR:DayType:12:" version="any"/>
<DayType id="FR:DayType:13:" version="any"/>
<DayType id="FR:DayType:14:" version="any"/>
<DayType id="FR:DayType:15:" version="any"/>
<DayType id="FR:DayType:16:" version="any"/>
<DayType id="FR:DayType:17:" version="any"/>
<DayType id="FR:DayType:18:" version="any"/>
<DayType id="FR:DayType:19:" version="any"/>
<DayType id="FR:DayType:20:" version="any"/>
<DayType id="FR:DayType:21:" version="any"/>
<DayType id="FR:DayType:22:" version

## Comment

The extraction produced a **238.3 MB** file, much larger than expected and too large to parse normally. Looking at the preview, the reason is clear: each `DayType` element is self-closing (`<DayType id="FR:DayType:1:" version="any"/>`) and contains **no child elements at all**. There is no name, no description, no weekday flags ,just an ID.

This means `DayType` in the France NeTEx feed is essentially just a label that gets linked to a `UicOperatingPeriod` via a `DayTypeAssignment`. All the actual calendar information lives in the `UicOperatingPeriod` table we already extracted.

Given that the `DayType` elements contain no useful information beyond their ID, and the file is 238 MB, **there is no need to parse this file**. The `UicOperatingPeriod` table with `ValidDayBits` is sufficient for the calendar comparison. 

# Part 3: GTFS and NeTEx Comparison

## 3.1 Station-level

## Preparing the coordinates for matching

Before matching, I make sure the coordinate columns are numeric on both sides. This is a necessary step because coordinates read from XML or CSV files are sometimes stored as text strings, which would cause errors in the distance calculation.

In [35]:
# Ensure coordinates are numeric on both sides
gtfs_station_stops["stop_lat"] = pd.to_numeric(gtfs_station_stops["stop_lat"], errors="coerce")
gtfs_station_stops["stop_lon"] = pd.to_numeric(gtfs_station_stops["stop_lon"], errors="coerce")

netex_stops_fr["latitude"]  = pd.to_numeric(netex_stops_fr["latitude"],  errors="coerce")
netex_stops_fr["longitude"] = pd.to_numeric(netex_stops_fr["longitude"], errors="coerce")

print("GTFS station stops:", len(gtfs_station_stops))
print("NeTEx StopPlaces:  ", len(netex_stops_fr))

GTFS station stops: 3506
NeTEx StopPlaces:   3506


## Comment

Both GTFS and NeTEx contain exactly **3,506 stations**.

This is an encouraging sign before the matching step.

If both feeds describe the same set of stations, a high match rate in the coordinate-based comparison would be expected.

In [36]:
# For each GTFS station, find the nearest NeTEx StopPlace
netex_coords = netex_stops_fr[["latitude", "longitude"]].values.tolist()
netex_ids    = netex_stops_fr["stopplace_id"].tolist()
netex_names  = netex_stops_fr["stopplace_name"].tolist()

results = []
for _, gtfs_row in gtfs_station_stops.iterrows():
    gtfs_point = (gtfs_row["stop_lat"], gtfs_row["stop_lon"])
    distances  = [
        haversine(gtfs_point, (lat, lon), unit=Unit.METERS)
        for lat, lon in netex_coords
    ]
    best_idx  = distances.index(min(distances))
    results.append({
        "stop_id":        gtfs_row["stop_id"],
        "stop_name":      gtfs_row["stop_name"],
        "stopplace_id":   netex_ids[best_idx],
        "stopplace_name": netex_names[best_idx],
        "distance_m":     distances[best_idx],
    })

station_match_fr = pd.DataFrame(results)

print(f"Total GTFS stations:        {len(station_match_fr):,}")
print(f"Matched within 50m:         {(station_match_fr['distance_m'] <= 50).sum():,}")
print(f"Matched within 100m:        {(station_match_fr['distance_m'] <= 100).sum():,}")
print(f"Unmatched (beyond 100m):    {(station_match_fr['distance_m'] > 100).sum():,}")

display(station_match_fr.head(10))

Total GTFS stations:        3,506
Matched within 50m:         3,506
Matched within 100m:        3,506
Unmatched (beyond 100m):    0


,stop_id,stop_name,stopplace_id,stopplace_name,distance_m
0,StopArea:OCE71043075,FIGUERES-VILAFANT,FR::LMO:72425e40-3a69-11e9-8417-bb1d8a705241:,FIGUERES-VILAFANT,0.0
1,StopArea:OCE71718010,Barcelone-Sants,FR::LMO:c249c6f0-fa0b-11e9-b755-b1228bd14244:,Barcelone-Sants,0.0
2,StopArea:OCE71793000,GIRONA,FR::LMO:7247b570-3a69-11e9-8417-bb1d8a705241:,GIRONA,0.0
3,StopArea:OCE71793150,Portbou,FR::LMO:c24903a0-fa0b-11e9-b755-b1228bd14244:,Portbou,0.0
4,StopArea:OCE80021402,Augsburg Hbf,FR::LMO:71425360-3a69-11e9-8417-bb1d8a705241:,Augsburg Hbf,0.0
5,StopArea:OCE80077990,Berlin-Gesundbrunnen,FR::LMO:71a3fb60-3a69-11e9-8417-bb1d8a705241:,Berlin-Gesundbrunnen,0.0
6,StopArea:OCE80110650,Frankfurt (Main) Sued,FR::LMO:71bb05d0-3a69-11e9-8417-bb1d8a705241:,Frankfurt (Main) Sued,0.0
7,StopArea:OCE80110684,Francfort sur le Main,FR::LMO:71ba90a0-3a69-11e9-8417-bb1d8a705241:,Francfort sur le Main,0.0
8,StopArea:OCE80140087,Mannheim Hbf,FR::LMO:71cdf190-3a69-11e9-8417-bb1d8a705241:,Mannheim Hbf,0.0
9,StopArea:OCE80140210,Heidelberg Hbf,FR::LMO:71d21040-3a69-11e9-8417-bb1d8a705241:,Heidelberg Hbf,0.0


## Output

The coordinate-based matching achieved a **100% match rate**: all 3,506 GTFS stations found a NeTEx StopPlace within 50 metres, and the unmatched count is zero.

Looking at the matched pairs, the result is even stronger than expected because every single pair has a distance of **0.0 metres**, meaning the coordinates are bit-for-bit identical across both feeds. The station names also match perfectly on both sides.

Even though the IDs are completely different between the two formats (`StopArea:OCE...` in GTFS vs `FR::LMO:uuid:` in NeTEx), the coordinates are identical, which is the strongest possible confirmation that both feeds describe the same physical stations.

## Conclusion: Station Comparison

Both feeds contain exactly **3,506 stations**. The coordinate-based matching achieved a **100% match rate**, with every GTFS station finding a NeTEx StopPlace at a distance of exactly **0.0 metres**. Station names are also identical on both sides.

Despite using completely different ID formats like `StopArea:OCE...` in GTFS and `FR::LMO:uuid:` in NeTEx, the coordinates are bit-for-bit identical, which confirms that both feeds are fully consistent at the station level.

## 3.2 Line-Level Comparison

I first look at both tables side by side to understand how the fields align and whether a direct match is possible.

In [37]:
print("GTFS routes sample:")
display(routes[["route_id", "route_short_name", "route_long_name"]].head(10))

print("NeTEx lines sample:")
display(netex_lines_fr[["line_id", "public_code", "line_name"]].head(10))

GTFS routes sample:


,route_id,route_short_name,route_long_name
0,FR:Line::00F2577A-6A87-42E0-95F3-07351E4BC2F6:,P53,Bening - Sarreguemines
1,FR:Line::00F7208C-CEBC-4521-A792-6EC3ABB65811:,C30,Saint-Étienne - Roanne
2,FR:Line::00e52513-07a0-4ce1-8b13-97cab39bdef8:,S3,Ligne S3 BreizhGo
3,FR:Line::0128E1D5-9183-4D58-B1CF-F5AA5A64A037:,C6,Marseille - Toulon - Hyeres
4,FR:Line::0202671B-7107-429E-A37B-473C55E0254C:,C8,Montpellier Saint-Roch - Avignon Centre
5,FR:Line::022B77D9-D121-4DCB-B808-FB2F7931866B:,C20,Nantes Cholet Angers Saumur
6,FR:Line::02534A5F-903C-454E-A24F-E6E2E23B3CBF:,P13,Ambérieu - Mâcon
7,FR:Line::0312E125-B6BC-404C-A7D1-C1600D4CACAF:,K1,Toulouse Matabiau - Brive La Gaillarde
8,FR:Line::03398D75-5B60-4588-BC05-9985E599BA20:,R16,Ussel - Bort Les Orgues
9,FR:Line::0359c353-2f81-4c49-a269-bc287ff237e0:,L22,22. Limoges - Uzerche - Brive


NeTEx lines sample:


,line_id,public_code,line_name
0,FR:Line::00F2577A-6A87-42E0-95F3-07351E4BC2F6:,P53,Bening - Sarreguemines
1,FR:Line::00F7208C-CEBC-4521-A792-6EC3ABB65811:,C30,Saint-Étienne - Roanne
2,FR:Line::00e52513-07a0-4ce1-8b13-97cab39bdef8:,S3,Ligne S3 BreizhGo
3,FR:Line::0128E1D5-9183-4D58-B1CF-F5AA5A64A037:,C6,Marseille - Toulon - Hyeres
4,FR:Line::0202671B-7107-429E-A37B-473C55E0254C:,C8,Montpellier Saint-Roch - Avignon Centre
5,FR:Line::022B77D9-D121-4DCB-B808-FB2F7931866B:,C20,Nantes Cholet Angers Saumur
6,FR:Line::02534A5F-903C-454E-A24F-E6E2E23B3CBF:,P13,Ambérieu - Mâcon
7,FR:Line::0312E125-B6BC-404C-A7D1-C1600D4CACAF:,K1,Toulouse Matabiau - Brive La Gaillarde
8,FR:Line::03398D75-5B60-4588-BC05-9985E599BA20:,R16,Ussel - Bort Les Orgues
9,FR:Line::0359c353-2f81-4c49-a269-bc287ff237e0:,L22,22. Limoges - Uzerche - Brive


## Output

The two tables are identical: the `route_id` in GTFS and `line_id` in NeTEx are exactly the same identifiers, and both `route_short_name`/`public_code` and `route_long_name`/`line_name` match perfectly row by row.

This means that unlike stations, where I had to rely on coordinate matching, the line-level comparison can be done with a **direct ID join**. Both feeds share the same line identifiers.

## Verifying ID overlap between GTFS and NeTEx

Since both tables looked identical in the preview, I verify whether the `route_id` in GTFS and the `line_id` in NeTEx are truly the same identifiers across all 728 rows not just in the first 10.

In [38]:
# Check if route_id in GTFS matches line_id in NeTEx
gtfs_ids   = set(routes["route_id"].tolist())
netex_ids  = set(netex_lines_fr["line_id"].tolist())

print("GTFS route IDs also in NeTEx:", len(gtfs_ids & netex_ids))
print("Only in GTFS:  ", len(gtfs_ids - netex_ids))
print("Only in NeTEx: ", len(netex_ids - gtfs_ids))

GTFS route IDs also in NeTEx: 728
Only in GTFS:   0
Only in NeTEx:  0


## Comment

All 728 GTFS route IDs are present in NeTEx, and all 728 NeTEx line IDs are present in GTFS with zero entries missing on either side. 

This is a stronger result than Luxembourg, where I had to extract a numeric ID from within the NeTEx string. Here SNCF uses the exact same identifier directly in both formats, making the line-level comparison trivial. Both feeds are fully consistent at the line level.

## Joining GTFS Routes and NeTEx Lines

Since both feeds share the same line IDs, I perform an outer join on `route_id` and `line_id` to formally confirm the match and compare the fields side by side. The `indicator=True` parameter adds a `_merge` column that shows whether each row came from both feeds, only GTFS, or only NeTEx.

In [39]:
# Join GTFS routes with NeTEx lines on the shared ID
line_match_fr = routes[["route_id", "route_short_name", "route_long_name", "route_type"]].merge(
    netex_lines_fr[["line_id", "public_code", "line_name", "transport_mode"]],
    left_on="route_id",
    right_on="line_id",
    how="outer",
    indicator=True
)

# Summary
n_gtfs    = routes["route_id"].nunique()
n_netex   = netex_lines_fr["line_id"].nunique()
n_matched = (line_match_fr["_merge"] == "both").sum()

print(f"Total GTFS routes:      {n_gtfs:,}")
print(f"Total NeTEx lines:      {n_netex:,}")
print(f"Matched:                {n_matched:,} ({round(n_matched/n_gtfs*100, 2)}%)")
print(f"Only in GTFS:           {(line_match_fr['_merge'] == 'left_only').sum():,}")
print(f"Only in NeTEx:          {(line_match_fr['_merge'] == 'right_only').sum():,}")

display(line_match_fr.head(10))

Total GTFS routes:      728
Total NeTEx lines:      728
Matched:                728 (100.0%)
Only in GTFS:           0
Only in NeTEx:          0


,route_id,route_short_name,route_long_name,route_type,line_id,public_code,line_name,transport_mode,_merge
0,FR:Line::00F2577A-6A87-42E0-95F3-07351E4BC2F6:,P53,Bening - Sarreguemines,3,FR:Line::00F2577A-6A87-42E0-95F3-07351E4BC2F6:,P53,Bening - Sarreguemines,coach,both
1,FR:Line::00F7208C-CEBC-4521-A792-6EC3ABB65811:,C30,Saint-Étienne - Roanne,2,FR:Line::00F7208C-CEBC-4521-A792-6EC3ABB65811:,C30,Saint-Étienne - Roanne,rail,both
2,FR:Line::00e52513-07a0-4ce1-8b13-97cab39bdef8:,S3,Ligne S3 BreizhGo,2,FR:Line::00e52513-07a0-4ce1-8b13-97cab39bdef8:,S3,Ligne S3 BreizhGo,rail,both
3,FR:Line::0128E1D5-9183-4D58-B1CF-F5AA5A64A037:,C6,Marseille - Toulon - Hyeres,2,FR:Line::0128E1D5-9183-4D58-B1CF-F5AA5A64A037:,C6,Marseille - Toulon - Hyeres,rail,both
4,FR:Line::0202671B-7107-429E-A37B-473C55E0254C:,C8,Montpellier Saint-Roch - Avignon Centre,2,FR:Line::0202671B-7107-429E-A37B-473C55E0254C:,C8,Montpellier Saint-Roch - Avignon Centre,rail,both
5,FR:Line::022B77D9-D121-4DCB-B808-FB2F7931866B:,C20,Nantes Cholet Angers Saumur,2,FR:Line::022B77D9-D121-4DCB-B808-FB2F7931866B:,C20,Nantes Cholet Angers Saumur,rail,both
6,FR:Line::02534A5F-903C-454E-A24F-E6E2E23B3CBF:,P13,Ambérieu - Mâcon,2,FR:Line::02534A5F-903C-454E-A24F-E6E2E23B3CBF:,P13,Ambérieu - Mâcon,rail,both
7,FR:Line::0312E125-B6BC-404C-A7D1-C1600D4CACAF:,K1,Toulouse Matabiau - Brive La Gaillarde,2,FR:Line::0312E125-B6BC-404C-A7D1-C1600D4CACAF:,K1,Toulouse Matabiau - Brive La Gaillarde,rail,both
8,FR:Line::03398D75-5B60-4588-BC05-9985E599BA20:,R16,Ussel - Bort Les Orgues,3,FR:Line::03398D75-5B60-4588-BC05-9985E599BA20:,R16,Ussel - Bort Les Orgues,coach,both
9,FR:Line::0359c353-2f81-4c49-a269-bc287ff237e0:,L22,22. Limoges - Uzerche - Brive,2,FR:Line::0359c353-2f81-4c49-a269-bc287ff237e0:,L22,22. Limoges - Uzerche - Brive,rail,both


## Output
The join confirms a **100% match rate**.

All 728 GTFS routes matched a NeTEx line, with zero rows only in GTFS and zero only in NeTEx. Every row in the merged table has `_merge = both`.

Looking at the matched pairs, the fields align perfectly on both sides. `route_short_name` matches `public_code`, and `route_long_name` matches `line_name` exactly. The `route_type` in GTFS and `transport_mode` in NeTEx also correspond correctly. For example `route_type = 2` (Rail) maps to `transport_mode = rail`, and `route_type = 3` (Bus/Coach) maps to `transport_mode = coach`.

This is the strongest possible result: both feeds are fully consistent at the line level, sharing the same IDs, names, and transport modes throughout.

In [40]:
# Transport mode comparison table: GTFS route_type vs NeTEx transport_mode
# (route_type_labels was already defined in the GTFS route_type distribution step
# above and is reused here — nothing between the two cells changes it)
transport_mode_compare = (
    line_match_fr.groupby(["route_type", "transport_mode"])
    .size()
    .reset_index(name="n_lines")
)

transport_mode_compare["route_type_label"] = transport_mode_compare["route_type"].map(route_type_labels)

transport_mode_compare = transport_mode_compare[
    ["route_type", "route_type_label", "transport_mode", "n_lines"]
].rename(columns={
    "route_type":       "GTFS route_type (code)",
    "route_type_label": "GTFS route_type (label)",
    "transport_mode":   "NeTEx transport_mode",
    "n_lines":          "Number of lines"
})

display(transport_mode_compare)

,GTFS route_type (code),GTFS route_type (label),NeTEx transport_mode,Number of lines
0,0,Tram,rail,4
1,2,Rail,coach,7
2,2,Rail,rail,564
3,2,Rail,unknown,20
4,3,Bus,coach,80
5,3,Bus,rail,9
6,3,Bus,unknown,44


## Comment

The table compares how each feed classifies the transport mode for the same 728 lines. The results show that while the line IDs match 100%, the transport mode classification is **not fully consistent** between the two feeds.

The largest group: **564 lines classified as Rail in GTFS and rail in NeTEx** matches perfectly. Beyond that, the picture is more nuanced:

- **GTFS Tram → NeTEx rail (4 lines)**: A genuine inconsistency. GTFS classifies these as tram (`route_type = 0`) while NeTEx calls them rail. Tram and rail are different modes, so the two feeds disagree here.

- **GTFS Rail → NeTEx coach (7 lines)**: Another real inconsistency. The same lines are classified as rail in GTFS but coach in NeTEx.

- **GTFS Rail → NeTEx unknown (20 lines)** and **GTFS Bus → NeTEx unknown (44 lines)**: These lines have a valid GTFS classification but no transport mode assigned in NeTEx.

- **GTFS Bus → NeTEx coach (80 lines)**: This is not a real inconsistency. GTFS uses `Bus` as a broad category while NeTEx uses the more specific term `coach`. They refer to the same concept expressed at different levels of detail.

- **GTFS Bus → NeTEx rail (9 lines)**: A genuine inconsistency. The two feeds classify the same lines as completely different modes.

Overall, the transport mode comparison reveals that while both feeds cover the same lines, the classification of modes is not always aligned. This is partly due to the two formats using different granularity, and partly due to genuine inconsistencies in how individual lines were labelled.

## Conclusion: Line Comparison

Both feeds contain exactly **728 lines** with a **100% match rate** based on a direct ID join. Both GTFS and NeTEx use the same line identifiers.

The line names and public codes are identical on both sides. However, the transport mode classification shows some inconsistencies. While the majority of lines agree, a small number of lines are classified differently between the two formats, either due to different levels of granularity or genuine labelling differences. The most notable case is 4 lines classified as Tram in GTFS but rail in NeTEx, and 9 lines classified as Bus in GTFS but rail in NeTEx.

## 3.3 Calendar-Level Comparison

## Preparing the GTFS Calendar Side

Since France has no `calendar.txt`, the service validity is defined entirely through `calendar_dates` with `exception_type = 1` meaning every entry is simply an added active date. There are no weekday flags to expand and no exceptions to subtract, which makes the GTFS side simpler than Luxembourg.

For each `service_id`, I collect all its active dates and encode them as a bit string over the full feed window (April 26 to September 23, 2026: 151 days), where `1` means the service runs on that day and `0` means it does not. This brings the GTFS calendar into the same format as the NeTEx `ValidDayBits`, making a direct comparison possible.

In [41]:
# Prepare GTFS calendar side from calendar_dates only
calendar_dates["date"] = pd.to_datetime(calendar_dates["date"].astype(str), format="%Y%m%d")

# Keep only exception_type = 1 (added dates)
calendar_dates_active = calendar_dates[calendar_dates["exception_type"] == 1].copy()

# Define the feed window
feed_dates = pd.date_range("2026-04-26", "2026-09-23")
print(f"Feed window: {feed_dates[0].date()} to {feed_dates[-1].date()}")
print(f"Total days in feed window: {len(feed_dates)}")

# Build bit string per service_id
gtfs_calendar_rows = []
for sid, group in calendar_dates_active.groupby("service_id"):
    active_dates = set(group["date"])
    bit_string   = "".join("1" if d in active_dates else "0" for d in feed_dates)
    gtfs_calendar_rows.append({
        "service_id":        sid,
        "n_active_days":     bit_string.count("1"),
        "first_active_date": min(active_dates),
        "last_active_date":  max(active_dates),
        "valid_day_bits":    bit_string,
    })

gtfs_calendar_fr = pd.DataFrame(gtfs_calendar_rows)

print(f"\nUnique GTFS service patterns: {len(gtfs_calendar_fr):,}")
print(f"Missing values:")
print(gtfs_calendar_fr.isna().sum())
display(gtfs_calendar_fr.head(10))

Feed window: 2026-04-26 to 2026-09-23
Total days in feed window: 151

Unique GTFS service patterns: 12,815
Missing values:
service_id           0
n_active_days        0
first_active_date    0
last_active_date     0
valid_day_bits       0
dtype: int64


,service_id,n_active_days,first_active_date,last_active_date,valid_day_bits
0,0,151,2026-04-26,2026-10-31,1111111111111111111111111111111111111111111111...
1,1,39,2026-07-06,2026-10-23,0000000000000000000000000000000000000000000000...
2,2,1,2026-07-13,2026-07-13,0000000000000000000000000000000000000000000000...
3,3,30,2026-07-06,2026-08-27,0000000000000000000000000000000000000000000000...
4,4,70,2026-04-26,2026-10-23,1000111100011110011111000011110001110000111000...
5,5,101,2026-04-27,2026-10-23,0111100011110001110000111110001111001111100111...
6,6,22,2026-04-27,2026-10-19,0100000010000001000000100000001000001000000100...
7,7,1,2026-05-13,2026-05-13,0000000000000000010000000000000000000000000000...
8,8,1,2026-05-20,2026-05-20,0000000000000000000000001000000000000000000000...
9,9,0,2026-10-06,2026-10-07,0000000000000000000000000000000000000000000000...


## Output

The GTFS side produces **12,815 unique service patterns** and there are zero missing values across all fields.

The feed window is **151 days** (April 26 to September 23, 2026), which matches the NeTEx validity window exactly.

A few interesting things are visible in the preview:
- `service_id = 0` runs for all **151 days**  likely a service that runs every single day of the feed window

- Some services run only **1 day** (e.g. service_id 2 and 7), suggesting special or one-off services

- Some `last_active_date` values extend beyond September 23 (e.g. `2026-10-31`). These dates fall outside the feed window and will be clipped to `0` in the bit string, which is correct behaviour.


## Comparing GTFS and NeTEx Calendar Patterns

To compare the two calendar representations, I first re-anchor the NeTEx `ValidDayBits` strings to the same 151-day feed window (April 26 to September 23, 2026) so that position 0 always means April 26 on both sides.

I then compute summary fields for the NeTEx side: `n_active_days`, `first_active_date`, and `last_active_date`  from the aligned bit string, making both sides directly comparable

After deduplicating both sides, I perform an outer merge on four fields together: `n_active_days`, `first_active_date`, `last_active_date`, and the bit string itself. Two patterns only count as a match if all four are identical.

One thing to flag: on the GTFS side, `first_active_date` and `last_active_date` come from the service's full calendar, which can extend past the comparison window (NeTEx's feed only runs to September 23, while GTFS continues into October). On the NeTEx side, these two dates always fall inside the window, since they are derived from the window-limited bit string. So a GTFS service that behaves identically to a NeTEx pattern within the shared window can still fail to match, simply because its last active date falls in October rather than September (see the note after the results below).


In [42]:
# Re-anchor NeTEx ValidDayBits to the same feed window
def realign_bits(from_date, valid_day_bits, feed_dates):
    if pd.isna(from_date) or not valid_day_bits:
        return None
    from_date = pd.Timestamp(from_date)
    result = []
    for d in feed_dates:
        offset = (d - from_date).days
        if 0 <= offset < len(valid_day_bits):
            result.append(valid_day_bits[offset])
        else:
            result.append("0")
    return "".join(result)

netex_operatingperiods_fr["from_date"] = pd.to_datetime(netex_operatingperiods_fr["from_date"])

netex_operatingperiods_fr["bits_aligned"] = netex_operatingperiods_fr.apply(
    lambda row: realign_bits(row["from_date"], row["valid_day_bits"], feed_dates),
    axis=1
)

# Deduplicate both sides on exact bit string fingerprint
compare_cols = ["n_active_days", "first_active_date", "last_active_date", "valid_day_bits"]

gtfs_patterns_unique = (
    gtfs_calendar_fr[compare_cols]
    .drop_duplicates()
    .copy()
)

netex_patterns_unique = (
    netex_operatingperiods_fr
    .assign(
        n_active_days     = netex_operatingperiods_fr["bits_aligned"].str.count("1"),
        first_active_date = netex_operatingperiods_fr.apply(
            lambda row: feed_dates[row["bits_aligned"].index("1")] if "1" in row["bits_aligned"] else pd.NaT, axis=1
        ),
        last_active_date  = netex_operatingperiods_fr.apply(
            lambda row: feed_dates[len(row["bits_aligned"]) - 1 - row["bits_aligned"][::-1].index("1")] if "1" in row["bits_aligned"] else pd.NaT, axis=1
        ),
        valid_day_bits    = netex_operatingperiods_fr["bits_aligned"]
    )
    [compare_cols]
    .drop_duplicates()
    .copy()
)

# Outer merge on exact fingerprint
calendar_pattern_compare_fr = gtfs_patterns_unique.merge(
    netex_patterns_unique,
    on=compare_cols,
    how="outer",
    indicator=True
)

# Summary
n_gtfs    = len(gtfs_patterns_unique)
n_netex   = len(netex_patterns_unique)
n_matched = (calendar_pattern_compare_fr["_merge"] == "both").sum()

print(f"Unique GTFS patterns:     {n_gtfs:,}")
print(f"Unique NeTEx patterns:    {n_netex:,}")
print(f"Matched patterns:         {n_matched:,}")
print(f"GTFS match rate:          {round(n_matched / n_gtfs * 100, 2)} %")
print(f"NeTEx match rate:         {round(n_matched / n_netex * 100, 2)} %")
print(f"\nOnly in GTFS:  {(calendar_pattern_compare_fr['_merge'] == 'left_only').sum():,}")
print(f"Only in NeTEx: {(calendar_pattern_compare_fr['_merge'] == 'right_only').sum():,}")

display(calendar_pattern_compare_fr.head(20))

Unique GTFS patterns:     12,277
Unique NeTEx patterns:    11,427
Matched patterns:         6,645
GTFS match rate:          54.13 %
NeTEx match rate:         58.15 %

Only in GTFS:  5,632
Only in NeTEx: 4,782


,n_active_days,first_active_date,last_active_date,valid_day_bits,_merge
0,0,2026-09-24,2026-09-24,0000000000000000000000000000000000000000000000...,left_only
1,0,2026-09-24,2026-10-02,0000000000000000000000000000000000000000000000...,left_only
2,0,2026-09-24,2026-10-13,0000000000000000000000000000000000000000000000...,left_only
3,0,2026-09-25,2026-09-25,0000000000000000000000000000000000000000000000...,left_only
4,0,2026-09-25,2026-10-02,0000000000000000000000000000000000000000000000...,left_only
5,0,2026-09-25,2026-10-09,0000000000000000000000000000000000000000000000...,left_only
6,0,2026-09-25,2026-10-23,0000000000000000000000000000000000000000000000...,left_only
7,0,2026-09-26,2026-09-26,0000000000000000000000000000000000000000000000...,left_only
8,0,2026-09-26,2026-09-27,0000000000000000000000000000000000000000000000...,left_only
9,0,2026-09-26,2026-10-03,0000000000000000000000000000000000000000000000...,left_only


## Sensitivity check: matching on the day-by-day pattern alone

The comparison above needs four things to match: active day count, first active date, last active date, and the actual pattern.

The problem: GTFS's first/last active dates can go past the shared window, while NeTEx's never do. So two services could run the exact same days inside the window and still count as different, just because GTFS's official end date is later

For example, a GTFS service might actually run until October 31, but NeTEx only covers up to September 23. Inside that window the two could match perfectly day for day, but the original check would still call it a mismatch because the end dates do not line up.

So here I repeat the comparison using only two things: active day count and the actual pattern within the window. I drop the first/last date fields, since those are exactly what caused the unfair mismatches above.

In [43]:
print("GTFS latest active date:", gtfs_calendar_fr["last_active_date"].max().date())
print("NeTEx window ends:", feed_dates[-1].date())

n_beyond = (gtfs_calendar_fr["last_active_date"] > feed_dates[-1]).sum()
print("GTFS services ending after the NeTEx window:", n_beyond, "out of", len(gtfs_calendar_fr))

GTFS latest active date: 2026-10-31
NeTEx window ends: 2026-09-23
GTFS services ending after the NeTEx window: 6170 out of 12815


GTFS last active date ends in October, while NeTEx last active date endds in September

In [44]:
# Sensitivity check: dedupe and merge using only n_active_days + valid_day_bits
# (reuses gtfs_calendar_fr and netex_operatingperiods_fr["bits_aligned"] computed above)
bits_only_cols = ["n_active_days", "valid_day_bits"]

gtfs_bits_only_unique = (
    gtfs_calendar_fr[bits_only_cols]
    .drop_duplicates()
    .copy()
)

netex_bits_only_unique = (
    netex_operatingperiods_fr
    .assign(
        n_active_days  = netex_operatingperiods_fr["bits_aligned"].str.count("1"),
        valid_day_bits = netex_operatingperiods_fr["bits_aligned"]
    )
    [bits_only_cols]
    .drop_duplicates()
    .copy()
)

bits_only_compare = gtfs_bits_only_unique.merge(
    netex_bits_only_unique,
    on=bits_only_cols,
    how="outer",
    indicator=True
)

n_gtfs_bits    = len(gtfs_bits_only_unique)
n_netex_bits   = len(netex_bits_only_unique)
n_matched_bits = (bits_only_compare["_merge"] == "both").sum()

print(f"Unique GTFS day-patterns:   {n_gtfs_bits:,}")
print(f"Unique NeTEx day-patterns:  {n_netex_bits:,}")
print(f"Matched day-patterns:       {n_matched_bits:,}")
print(f"GTFS match rate:            {round(n_matched_bits / n_gtfs_bits * 100, 2)} %")
print(f"NeTEx match rate:           {round(n_matched_bits / n_netex_bits * 100, 2)} %")

Unique GTFS day-patterns:   11,204
Unique NeTEx day-patterns:  11,427
Matched day-patterns:       11,203
GTFS match rate:            99.99 %
NeTEx match rate:           98.04 %


## Output

The comparison yields **6,645 matched patterns** out of 12,277 unique GTFS patterns and 11,427 unique NeTEx patterns, giving a **GTFS match rate of 54.13%** and a **NeTEx match rate of 58.15%**.

This is a lower match rate than Luxembourg (81%), but this is expected for several reasons:

- The feed window is **151 days** instead of 23 days, meaning there are far more possible combinations of active days, making exact bit-level matches harder to achieve

- GTFS has **12,815 service patterns** covering a large national network, while NeTEx has **11,427 operating periods**. The two formats model service validity at different levels of granularity, similar to what we saw in Luxembourg

- Many of the unmatched GTFS patterns have `last_active_date` values beyond September 23 (e.g. `2026-10-31`), which fall outside the NeTEx feed window. These dates get clipped to `0` in the GTFS bit string but may be encoded differently in NeTEx, causing mismatches

- **A note on what this number measures:** the 54.13%/58.15% figures above require GTFS and NeTEx patterns to agree on active-day count, first active date, last active date, *and* the day-by-day pattern. The sensitivity check above (matching on `n_active_days` and `valid_day_bits` only, dropping the two asymmetric date fields) shows a **GTFS match rate of 99.99%** and a **NeTEx match rate of 98.04%** (11,203 matched out of 11,204 unique GTFS day-patterns and 11,427 NeTEx day-patterns). This gap is mostly explained by many GTFS services continuing to run past NeTEx's September 23 cutoff, which pushes their last active date outside the shared window even when their day-by-day pattern inside the window is identical. The 54.13%/58.15% figures are the ones used for the thesis comparison and are kept as originally computed.

Overall, given the scale and complexity of the SNCF network and the much longer feed window, a match rate above 50% on an exact bit-level comparison is a reasonable result.

## Conclusion: Calendar Comparison

The calendar comparison achieved a **GTFS match rate of 54.13%** and a **NeTEx match rate of 58.15%**, with 6,645 patterns matching exactly out of 12,277 unique GTFS patterns and 11,427 unique NeTEx patterns.

The lower match rate compared to Luxembourg is expected. The feed window is much longer (151 days vs 23 days), which means far more possible combinations of active days and makes exact bit-level matches harder to achieve. Additionally, some GTFS services have active dates beyond the NeTEx feed window; because the matching criterion also requires exact agreement on first and last active date (not just the day-by-day pattern), this pushes the reported rate well below what a pure day-by-day comparison shows. The sensitivity check earlier in this section, which drops the two date fields and matches on `n_active_days` and `valid_day_bits` alone, gives a GTFS match rate of 99.99% and a NeTEx match rate of 98.04%.

Despite this, a match rate above 50% on an exact bit-level comparison across a national network of this scale is a reasonable result and confirms that both feeds are broadly consistent at the calendar level.

# 3.4 Summary: France GTFS vs NeTEx Comparison

The comparison between the France GTFS and NeTEx feeds produced strong results across all three levels:

**Stations:** Both feeds contain exactly 3,506 stations. Coordinate-based matching achieved a 100% match rate with a distance of exactly 0.0 metres for every pair

**Lines:** Both feeds contain exactly 728 lines sharing the same identifiers, resulting in a 100% match rate on a direct ID join. Line names and public codes are identical on both sides. Minor inconsistencies were found in transport mode classification for a small number of lines.

**Calendar:** The comparison achieved a 54% GTFS match rate and 58% NeTEx match rate under a strict matching criterion that requires agreement on active-day count, first/last active date, and the day-by-day pattern together. A sensitivity check limiting the comparison to the day-by-day pattern alone (Part 3.3) raises this to 99.99%/98.04%, showing that the lower headline figure mostly reflects the stricter criterion and GTFS's longer feed tail (through October vs NeTEx's September cutoff) rather than real inconsistencies between the two feeds